In [5]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline
words = open("names.txt", "r").read().splitlines()

In [6]:
chars = sorted(list(set(''.join(words))))
chars = ['.'] + chars
stoi = {str:ix for ix, str in enumerate(chars)}
itos = {ix: str for str, ix in stoi.items()} 

Although we successful built a bigram model in the other notebook based on the counts of every possible combination in the 27 possible letters (including .), there are a few flaws that come with it. 

Firstly, if we are going to predict the next letter not only based on the single previous character, but not all (or a few) the characters that come before it, then we will need a table with much more than 2 dimension. In other words, this way of "counting" the frequency of each combination is not very scalable. 

Secondly, (of course we are learning nn) this way is not using nn for output. Therefore, we need to think of a way that somehow integrate the things we learnt about micrograd into it. We need to initialize a bunch of parameters that somehow takes a vector and produce an output that stands for a character just like the other notebook; we need to figure out a way to define a loss function whose output is a single number; we need to forward pass the parameters to obtain a loss function and use the backward pass to obtain the gradient of each parameter we initialize at first so that we can update them by moving them opposite the gradient. At the end of this, we want to obtain a set of parameters that minimize the loss function. However, we should not let the result of the loss function go below the loss function of the counting way since we still want some probablity for those combination that do not appear in the dataset, but not to merely minimizing the loss function. 

# Loss function and initializing parameters

By the counting method in the otehr notebook, we have produced a tensor with the possibility for each char to follow some given previous char among the 27 char. Here, we can just one hot encode each 27 possible char into 27 distinct tensor and use a randomly initialized 27*27 matrix to produce another tensor with a number assigned for each char following the previous char. (the higher the number, the higher the change it follows)

And then, for the loss function, we can apply exp() to each entries of tensor obtained by multiplying the one hot encoded matrix and parameter matrix, normoalizating each row to get a probability for each combination; then we iterate through the present combination in dataset, obtain the probablity for each combination that is correct; then we take the negative log of each probablity and mean in the end. Thus, we obtain a loss function then reflects better result when the output is low. 

In [3]:
# initializa the parameter matrix
g = torch.Generator().manual_seed(2147483647)
W = torch.rand((27, 27), generator=g, requires_grad = True)
W

tensor([[0.7081, 0.3542, 0.1054, 0.5996, 0.0904, 0.0899, 0.8822, 0.9887, 0.0080,
         0.2908, 0.7408, 0.4012, 0.8640, 0.7391, 0.3845, 0.2176, 0.7054, 0.9882,
         0.9600, 0.4388, 0.7074, 0.0554, 0.8117, 0.3059, 0.9928, 0.3419, 0.2070],
        [0.6071, 0.8581, 0.3224, 0.5998, 0.1621, 0.3729, 0.5870, 0.3176, 0.3409,
         0.9189, 0.7568, 0.6050, 0.5705, 0.7584, 0.9721, 0.5651, 0.4407, 0.6008,
         0.1163, 0.4279, 0.0574, 0.0711, 0.8033, 0.3436, 0.4670, 0.1440, 0.5037],
        [0.4244, 0.8139, 0.2275, 0.4577, 0.7701, 0.8770, 0.0389, 0.5286, 0.7724,
         0.6581, 0.6266, 0.4569, 0.8323, 0.9779, 0.3174, 0.4722, 0.3212, 0.6840,
         0.8664, 0.6026, 0.9374, 0.1093, 0.4679, 0.1706, 0.8736, 0.0081, 0.7697],
        [0.7052, 0.5996, 0.6095, 0.5802, 0.7099, 0.5512, 0.6437, 0.6311, 0.1029,
         0.7180, 0.4829, 0.0205, 0.7829, 0.9025, 0.3525, 0.8432, 0.0964, 0.9262,
         0.3013, 0.8271, 0.3661, 0.9071, 0.1658, 0.6749, 0.6350, 0.6475, 0.7717],
        [0.6585, 0.4204,

In [12]:
# all the present combination (index of ch1, and index of ch2) in the dataset
xs, ys = [], []
for word in words:
    chs = ['.'] + list(word) + ['.']
    for ch1, ch2, in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])
xs = torch.tensor(xs)
ys = torch.tensor(ys)
print(xs[:10])
print(ys[:10])
print(len(xs))

tensor([ 0,  5, 13, 13,  1,  0, 15, 12,  9, 22])
tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9])
228146


In [13]:
# one-hot encoding xs into a 228146 * 27 matrix
import torch.nn.functional as F
xenc = F.one_hot(xs, num_classes = 27).float() # using float since the result after a few steps involving exp() and normalization should be accurate
xenc[:10, :]

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0

In [16]:
# obtaining the matrix representing the probablity of next char
log_counts = xenc @ W # shape 228146 * 27 
# the matrix multiplication gives the matrix that represents log of the count 
counts = log_counts.exp() # shape 228146 * 27
# obtaining the matrix representing the counts
probs = counts / counts.sum(1, keepdims = True) # shape 228146 * 27
# normalizing each row to obtain the matrix representing probability
probs[:3, :]

tensor([[0.0426, 0.0299, 0.0233, 0.0382, 0.0229, 0.0229, 0.0507, 0.0563, 0.0211,
         0.0280, 0.0440, 0.0313, 0.0497, 0.0439, 0.0308, 0.0261, 0.0424, 0.0563,
         0.0547, 0.0325, 0.0425, 0.0222, 0.0472, 0.0285, 0.0566, 0.0295, 0.0258],
        [0.0298, 0.0278, 0.0241, 0.0308, 0.0300, 0.0362, 0.0401, 0.0269, 0.0538,
         0.0445, 0.0560, 0.0239, 0.0358, 0.0258, 0.0370, 0.0542, 0.0275, 0.0513,
         0.0263, 0.0253, 0.0315, 0.0294, 0.0317, 0.0507, 0.0554, 0.0562, 0.0380],
        [0.0317, 0.0341, 0.0391, 0.0342, 0.0347, 0.0217, 0.0243, 0.0281, 0.0464,
         0.0537, 0.0481, 0.0261, 0.0520, 0.0339, 0.0449, 0.0443, 0.0228, 0.0305,
         0.0345, 0.0543, 0.0219, 0.0490, 0.0287, 0.0234, 0.0548, 0.0312, 0.0516]],
       grad_fn=<SliceBackward0>)

In [21]:
# obtain the loss function
correct_probs = probs[torch.arange(len(xs)), ys]
# obtaining the probability predicted for the correct char
loss = -correct_probs.log().mean()
loss.item()

3.359710931777954

# Forward & Backward pass + Update

In [26]:
# setting seed and initializing first
g = torch.Generator().manual_seed(2147483647)
W = torch.rand((27, 27), generator=g, requires_grad = True)

for i in range(1000):
    # forward pass
    log_counts = xenc @ W 
    counts = log_counts.exp() 
    probs = counts / counts.sum(1, keepdims = True)
    correct_probs = probs[torch.arange(len(xs)), ys]
    loss = -correct_probs.log().mean()
    print(f"at round {i+1}, the loss is {loss.item()}")

    # backward pass
    W.grad = None # setting the grad of each entry on parameter matrix to None
    loss.backward()

    # update the parameter matrix
    W.data -= 50 * W.grad



at round 1, the loss is 3.359710931777954
at round 2, the loss is 3.100273847579956
at round 3, the loss is 2.941926956176758
at round 4, the loss is 2.8439781665802
at round 5, the loss is 2.776561737060547
at round 6, the loss is 2.728274345397949
at round 7, the loss is 2.69258975982666
at round 8, the loss is 2.6652886867523193
at round 9, the loss is 2.6437065601348877
at round 10, the loss is 2.626169204711914
at round 11, the loss is 2.611603260040283
at round 12, the loss is 2.5992932319641113
at round 13, the loss is 2.588744640350342
at round 14, the loss is 2.5796008110046387
at round 15, the loss is 2.5715978145599365
at round 16, the loss is 2.5645344257354736
at round 17, the loss is 2.5582549571990967
at round 18, the loss is 2.55263614654541
at round 19, the loss is 2.5475807189941406
at round 20, the loss is 2.5430078506469727
at round 21, the loss is 2.5388522148132324
at round 22, the loss is 2.5350587368011475
at round 23, the loss is 2.531582832336426
at round 24, 

# Apply loss function on counting method

In [25]:
# copying the two way table from the other ipynb
N = torch.zeros((27, 27), dtype = torch.int32)
for word in words:
    chars = ['.'] + list(word) + ['.']
    for ch1, ch2 in zip(chars, chars[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1
P  = N.float()/N.sum(1, keepdims = True)
P[:3, :]

# obtaining the value of loss function
correct_probs = P[xs, ys]
loss = -correct_probs.log().mean()
loss.item()

2.454014301300049

However, there is theoretically a case in which with a mocdel we create a combination that never appears in the dataset take places, with probablity of 0, which will result in a positive infinity for loss. 

Therefore, to prevent that case, we add 1 to each count to each entry of N so that no infinity will cover. (This process is called smoothing)

Of course, one can add any positive number other than 1. However, it needs to be noted that the more you add, the higher the loss function is going to get, since the more one add, the probablity is going to be more uniform, lowering the probablity of frequent combination in the dataset. (you can try by adding numbers different from 1.)

In [27]:
N = torch.zeros((27, 27), dtype = torch.int32)
for word in words:
    chars = ['.'] + list(word) + ['.']
    for ch1, ch2 in zip(chars, chars[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1
N = N + 1
P  = N.float()/N.sum(1, keepdims = True)
P[:3, :]

# obtaining the value of loss function
correct_probs = P[xs, ys]
loss = -correct_probs.log().mean()
loss.item()

2.4545767307281494

And if this process of smoothing is necessary in the counting method, it needs to be used somewhere in the neural network method too. 

Note that the goal of iterating the process of updating W is to minimize the loss function, W may change itself in a way that the probability produced gives 0 for those combinations that never appear in order to maximize the probability of those appearing frequently.

Therefore, we need to add a term in the loss function that encourages the probablities to be uniform: *c * (W**2).mean()*, where c is a constant, i.e., the scattered the entries of W are, the higher the loss function. 

The higher c is, the more encouraged the uniform values and therefore the uniform probabilities are. 

In [ ]:
# setting seed and initializing first
g = torch.Generator().manual_seed(2147483647)
W = torch.rand((27, 27), generator=g, requires_grad = True)

for i in range(1000):
    # forward pass
    log_counts = xenc @ W 
    counts = log_counts.exp() 
    probs = counts / counts.sum(1, keepdims = True)
    correct_probs = probs[torch.arange(len(xs)), ys]
    loss = -correct_probs.log().mean() + 0.01 * (W**2).mean()
    print(f"at round {i+1}, the loss is {loss.item()}")

    # backward pass
    W.grad = None # setting the grad of each entry on parameter matrix to None
    loss.backward()

    # update the parameter matrix
    W.data -= 50 * W.grad
    

at round 1, the loss is 3.3630006313323975
at round 2, the loss is 3.1034915447235107
at round 3, the loss is 2.9455957412719727
at round 4, the loss is 2.848206043243408
at round 5, the loss is 2.7813658714294434
at round 6, the loss is 2.7336270809173584
at round 7, the loss is 2.698446750640869
at round 8, the loss is 2.671607494354248
at round 9, the loss is 2.650451898574829
at round 10, the loss is 2.6333112716674805
at round 11, the loss is 2.6191165447235107
at round 12, the loss is 2.60715651512146
at round 13, the loss is 2.596938371658325
at round 14, the loss is 2.5881080627441406
at round 15, the loss is 2.5804030895233154
at round 16, the loss is 2.5736241340637207
at round 17, the loss is 2.5676162242889404
at round 18, the loss is 2.562257766723633
at round 19, the loss is 2.5574519634246826
at round 20, the loss is 2.5531182289123535
at round 21, the loss is 2.5491929054260254
at round 22, the loss is 2.545621633529663
at round 23, the loss is 2.5423591136932373
at rou

In [29]:
W

tensor([[-3.3664e+00,  1.9341e+00,  7.1697e-01,  8.8284e-01,  9.7441e-01,
          8.7568e-01, -4.1372e-01,  5.1582e-02,  3.1680e-01, -7.0992e-02,
          1.3343e+00,  1.5360e+00,  9.0208e-01,  1.3811e+00,  5.8660e-01,
         -4.6911e-01, -2.0668e-01, -1.8092e+00,  9.4379e-01,  1.1699e+00,
          7.1849e-01, -1.9442e+00, -5.1467e-01, -7.1108e-01, -1.4844e+00,
         -1.6917e-01,  3.7750e-01],
        [ 2.5059e+00,  2.9128e-02,  2.1221e-03, -1.3651e-01,  6.5300e-01,
          2.4578e-01, -1.3292e+00, -1.1233e+00,  1.4582e+00,  1.1121e+00,
         -1.0856e+00,  5.0227e-02,  1.5390e+00,  1.1023e+00,  2.3059e+00,
         -1.9612e+00, -1.7521e+00, -1.9983e+00,  1.7948e+00,  7.2323e-01,
          2.3858e-01, -3.4227e-01,  4.3122e-01, -1.1625e+00, -1.0491e+00,
          1.3292e+00, -2.1253e-01],
        [ 1.4876e+00,  2.5522e+00,  4.1604e-01, -1.0373e+00,  9.2328e-01,
          3.2832e+00, -1.1110e+00, -1.1010e+00,  4.8498e-01,  2.1490e+00,
         -1.0348e+00, -1.1023e+00,  1.38

# Generating a few words with NN

In [31]:
g = torch.Generator().manual_seed(2147483647)
# generating 5 words
for _ in range(5):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        log_counts = xenc @ W 
        counts = log_counts.exp() 
        probs = counts / counts.sum(1, keepdims = True)
        ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(f"The word generated is {''.join(out)}")

The word generated is cexze.
The word generated is momasurailezityha.
The word generated is konimittain.
The word generated is llayn.
The word generated is ka.


The result should be similar to the result of the counting method, which is expected since the NN way is just the counting method implemented in another form that is much more scalable and flexible.

Explicity, a tensor allows one to put the information about all the chars appearing before the char to be predicted, and then all those factors can be taken into consideration when conducting the matrix multiplication, which is assumed to be the content of the next lecture